In [42]:
import pandas as pd
import numpy as np
import time
import joblib
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, f1_score, cohen_kappa_score 

In [43]:
X_train = pd.read_csv("dataset/processed/X_train_featured.csv")
X_test = pd.read_csv("dataset/processed/X_test_featured.csv")
y_train = pd.read_csv("dataset/processed/y_train.csv")
y_test = pd.read_csv("dataset/processed/y_test.csv")

print("\n" + "="*40)
print("DATA SHAPE RESULTS")
print("="*40)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print("-" * 40)
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")


DATA SHAPE RESULTS
X_train shape: (2492096, 30)
y_train shape: (2492096, 1)
----------------------------------------
X_test shape:  (623019, 30)
y_test shape:  (623019, 1)


In [45]:
# Table to store results
results_table = []

def evaluate_regressor(model, X_train, y_train, X_test, y_test, model_name):
    # 1. PREDICTION (Predict continuous values)
    y_train_pred_raw = model.predict(X_train)
    y_test_pred_raw = model.predict(X_test)
    
    # 2. PROCESSING (Clip values between 1-4 and round for classification metrics)
    y_train_pred_class = np.round(np.clip(y_train_pred_raw, 1, 4)).astype(int)
    y_test_pred_class = np.round(np.clip(y_test_pred_raw, 1, 4)).astype(int)
    
    # 3. COMPUTE METRICS
    
    # MAE (regression metric)
    train_mae = mean_absolute_error(y_train, y_train_pred_raw)
    test_mae = mean_absolute_error(y_test, y_test_pred_raw)
    
    # F1-score (classification metric)
    train_f1 = f1_score(y_train, y_train_pred_class, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred_class, average='weighted')
    
    # Quadratic Weighted Kappa (ordinal metric)
    train_qwk = cohen_kappa_score(y_train, y_train_pred_class, weights='quadratic')
    test_qwk = cohen_kappa_score(y_test, y_test_pred_class, weights='quadratic')
    
    # 4. PRINT RESULTS
    print(f"✅ {model_name} Results:")
    print(f"   [TRAIN] MAE: {train_mae:.4f} | F1: {train_f1:.4f} | QWK: {train_qwk:.4f}")
    print(f"   [TEST]  MAE: {test_mae:.4f} | F1: {test_f1:.4f} | QWK: {test_qwk:.4f}")
    print(f"   -> Gap (Overfitting): {train_qwk - test_qwk:.4f} (QWK diff), {train_f1 - test_f1:.4f} (F1 diff)")
    print("-" * 60)
    
    # 5. RETURN DICTIONARY
    return {
        'Model': model_name,
        'MAE': test_mae,
        'F1': test_f1,
        'QWK': test_qwk,
        'Object': model
    }

### `evaluate_regressor` Function

This function evaluates a regression model for an **ordinal classification task** (severity 1–4) using the most relevant metrics: **MAE, F1-score (weighted), and Quadratic Weighted Kappa (QWK)**.

## Steps

1. **Prediction**  
   Generate raw predictions on training and test datasets.

2. **Processing**  
   Clip predictions to the range `[1, 4]` and round them to compute classification-like metrics.

3. **Metrics Computation**  
   - **MAE (Mean Absolute Error):** Measures the average deviation of predictions from true values in severity levels.  
   - **F1-score (weighted):** Measures the balance between precision and recall, accounting for class imbalance.  
   - **Quadratic Weighted Kappa (QWK):** Evaluates ordinal agreement, penalizing predictions more when they are farther from the true severity.

4. **Results Display**  
   Print training and test metrics, including the F1 gap to indicate potential overfitting.

5. **Return Value**  
   Return a dictionary containing test metrics (`MAE`, `F1`, `QWK`) and the fitted model object for comparison or saving.

In [46]:
print("🌲 Running Random Forest Regressor")
start = time.time()

# 1. Initialize
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# 2. Train
rf_model.fit(X_train, y_train.values.ravel())

# 3. Evaluate
results_table.append(evaluate_regressor(rf_model, X_train, y_train.values.ravel(), X_test, y_test.values.ravel(), "Random Forest"))
print(f"⏱️ Time: {time.time()-start:.2f}s")

🌲 Running Random Forest Regressor
✅ Random Forest Results:
   [TRAIN] MAE: 0.1528 | F1: 0.9075 | QWK: 0.2032
   [TEST]  MAE: 0.1564 | F1: 0.9052 | QWK: 0.1668
   -> Gap (Overfitting): 0.0364 (QWK diff), 0.0023 (F1 diff)
------------------------------------------------------------
⏱️ Time: 407.82s


In [47]:
print("🚀 Running XGBoost Regressor")
start = time.time()

# 1. Initialize with fixed parameters
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# 2. Train
xgb_model.fit(X_train, y_train)

# 3. Evaluate
results_table.append(evaluate_regressor(xgb_model, X_train, y_train, X_test, y_test, "XGBoost"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

🚀 Running XGBoost Regressor
✅ XGBoost Results:
   [TRAIN] MAE: 0.1549 | F1: 0.9053 | QWK: 0.1939
   [TEST]  MAE: 0.1576 | F1: 0.9041 | QWK: 0.1681
   -> Gap (Overfitting): 0.0258 (QWK diff), 0.0013 (F1 diff)
------------------------------------------------------------
⏱️ Time: 22.71s


In [48]:
print("⚡ Running LightGBM Regressor")
start = time.time()

# 1. Initialize
lgbm_model = LGBMRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# 2. Train
lgbm_model.fit(X_train, y_train)

# 3. Evaluate
results_table.append(evaluate_regressor(lgbm_model, X_train, y_train, X_test, y_test, "LightGBM"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

⚡ Running LightGBM Regressor
✅ LightGBM Results:
   [TRAIN] MAE: 0.1548 | F1: 0.9057 | QWK: 0.1937
   [TEST]  MAE: 0.1575 | F1: 0.9047 | QWK: 0.1712
   -> Gap (Overfitting): 0.0225 (QWK diff), 0.0010 (F1 diff)
------------------------------------------------------------
⏱️ Time: 19.14s


In [49]:
print("🐱 Running CatBoost Regressor")
start = time.time()

# 1. Initialize
cat_model = CatBoostRegressor(
    n_estimators=50,
    random_state=42,
    verbose=0
)

# 2. Train
cat_model.fit(X_train, y_train)

# 3. Evaluate
results_table.append(evaluate_regressor(cat_model, X_train, y_train, X_test, y_test, "CatBoost"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

🐱 Running CatBoost Regressor
✅ CatBoost Results:
   [TRAIN] MAE: 0.1550 | F1: 0.9059 | QWK: 0.2005
   [TEST]  MAE: 0.1581 | F1: 0.9042 | QWK: 0.1716
   -> Gap (Overfitting): 0.0289 (QWK diff), 0.0017 (F1 diff)
------------------------------------------------------------
⏱️ Time: 20.77s


In [50]:
# 1. Create comparison DataFrame
df_results = pd.DataFrame(results_table)

# Sort by QWK (highest first) — prioritize ordinal agreement
df_results = df_results.sort_values(by='QWK', ascending=False)

# Columns to display
display_cols = ['Model', 'MAE', 'F1', 'QWK']

print("\n🏆 MODEL LEADERBOARD")
print(df_results[display_cols].to_markdown(index=False))

# 2. Select the best model based on QWK
best_model_info = df_results.iloc[0]
best_model_name = best_model_info['Model']
best_model_object = best_model_info['Object']

print(f"\n🥇 WINNER: {best_model_name}")
print(f"   with QWK: {best_model_info['QWK']:.4f}, F1-Score: {best_model_info['F1']:.4f}, MAE: {best_model_info['MAE']:.4f}")

# 3. Save the best model
filename = f"best_model_{best_model_name.replace(' ', '_')}.pkl"
joblib.dump(best_model_object, filename)
print(f"💾 Saved the best model to file: {filename}")


🏆 MODEL LEADERBOARD
| Model         |      MAE |       F1 |      QWK |
|:--------------|---------:|---------:|---------:|
| CatBoost      | 0.158136 | 0.904243 | 0.171617 |
| LightGBM      | 0.157534 | 0.904706 | 0.171201 |
| XGBoost       | 0.157645 | 0.904063 | 0.168051 |
| Random Forest | 0.15642  | 0.905187 | 0.166802 |

🥇 WINNER: CatBoost
   with QWK: 0.1716, F1-Score: 0.9042, MAE: 0.1581
💾 Saved the best model to file: best_model_CatBoost.pkl


## 📈 Performance Summary & Strategic Insights
---

## 1. 🚀 Optimal Choice: CatBoost Regressor

**Decision:** CatBoost is selected as the primary model.

**Key Strength:**  
CatBoost achieved the **highest QWK (0.1716)** among all candidates.  
For an ordinal problem like severity prediction (1–4), this shows that the model *understands the severity order* and minimizes high-impact misclassifications (e.g., confusing very severe cases with very mild ones). This capability is crucial in safety-critical applications.

---

## 2. ⚖️ The Random Forest Paradox (High F1 but Low QWK)

**Observation:**  
Random Forest obtained the **highest F1-score (0.9052)** and the **lowest MAE (0.1564)**, yet ranked **last in QWK (0.1668)**.

**Explanation:**  
This is a classic sign of **majority-class bias**.  
Random Forest adopts a “safe” strategy: predicting most samples as the dominant severity class (Severity 2).  

This boosts MAE and weighted F1 (because the majority class dominates the metric), but fails to distinguish rare but important severe cases — leading to low QWK.

---

## 3. 🌟 Strength of Boosting Models (CatBoost, LightGBM, XGBoost)

Boosting-based models demonstrated strong capability in capturing:

- complex relationships,
- ordinal behavior,
- subtle distinctions between severity levels.

The difference between CatBoost and LightGBM is *minimal*, showing both are **high-quality, top-tier** solutions.  
CatBoost is preferred due to its stable handling of categorical features and overall consistency.

---

## 💡 Growth Opportunities (Future Roadmap)

The current QWK (~0.17) reflects the inherent complexity of real-world accident severity data.  
This is **not** a limitation — it is the foundation for future improvements.

What we already have:
- A **very strong baseline**, with F1 reaching **~90%**.

Next steps to improve QWK:
- deeper **feature engineering** (interaction features, domain-based transformations),  
- advanced **sampling techniques** to help the model learn rare severe cases better,  
- tuning models specifically for ordinal objectives.

---

## 🎯 Conclusion

Choosing **CatBoost** is a strategic decision — prioritizing real-world effectiveness, safety, and correct severity understanding.  
This choice provides a **solid foundation** for future enhancements and aligns perfectly with the project’s practical goals.

---